<a href="https://colab.research.google.com/github/Tamannapanwar17/deep-learning-cnn/blob/main/age_gender_revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [ ]:
import os
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array


class UTKGenerator(Sequence):

    def __init__(
        self,
        dataframe,
        folder_path,
        batch_size=32,
        target_size=(200, 200),
        augment=False,
        shuffle=True
    ):
        self.df = dataframe.reset_index(drop=True)
        self.folder_path = folder_path
        self.batch_size = batch_size
        self.target_size = target_size
        self.augment = augment
        self.shuffle = shuffle

        self.indexes = np.arange(len(self.df))

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):

        indexes = self.indexes[
            index * self.batch_size:
            (index + 1) * self.batch_size
        ]

        batch = self.df.iloc[indexes]

        images = []
        ages = []
        genders = []

        for _, row in batch.iterrows():

            image_path = os.path.join(
                self.folder_path,
                row['img']
            )

            image = load_img(
                image_path,
                target_size=self.target_size
            )

            image = img_to_array(image)

            if self.augment:
                image = tf.image.random_flip_left_right(image)
                image = tf.image.random_brightness(image, 0.1)

            image = image / 255.0

            images.append(image)
            ages.append(row['age'])
            genders.append(row['gender'])

        images = np.array(images, dtype=np.float32)

        ages = np.array(
            ages,
            dtype=np.float32
        ).reshape(-1, 1)

        genders = np.array(
            genders,
            dtype=np.float32
        ).reshape(-1, 1)

        return images, {
            'age': ages,
            'gender': genders
        }

    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
len(age)

23708

In [ ]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [ ]:
df.shape

(23708, 3)

In [ ]:
df.head()

,age,gender,img
0,65,0,65_0_0_20170117162901325.jpg.chip.jpg
1,45,1,45_1_0_20170117193323733.jpg.chip.jpg
2,91,1,91_1_2_20170105174644734.jpg.chip.jpg
3,58,0,58_0_3_20170112205528628.jpg.chip.jpg
4,70,1,70_1_0_20170113003731381.jpg.chip.jpg


In [ ]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [ ]:
train_df.shape

(20000, 3)

In [ ]:
test_df.shape

(3708, 3)

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
class MultiOutputGenerator:
    def __init__(self, generator):
        self.generator = generator

    def __len__(self):
        return len(self.generator)

    def __getitem__(self, index):
        x, y = self.generator[index]
        return x, tuple(y)

    def on_epoch_end(self):
        self.generator.on_epoch_end()

In [ ]:
train_generator = UTKGenerator(
    train_df,
    folder_path,
    batch_size=32,
    target_size=(200, 200),
    augment=True,
    shuffle=True
)

test_generator = UTKGenerator(
    test_df,
    folder_path,
    batch_size=32,
    target_size=(200, 200),
    augment=False,
    shuffle=False
)

In [ ]:
x, y = train_generator[0]

print(x.shape)
print(y['age'].shape)
print(y['gender'].shape)

(32, 200, 200, 3)
(32, 1)
(32, 1)


In [ ]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [ ]:
model = Model(
    inputs=resnet.input,
    outputs={
        'age': output1,
        'gender': output2
    }
)

In [ ]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':1})

In [ ]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator
)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 148s 214ms/step - age_loss: 15.1250 - age_mae: 15.1250 - gender_accuracy: 0.5424 - gender_loss: 0.8396 - loss: 15.9645 - val_age_loss: 14.4027 - val_age_mae: 14.4060 - val_gender_accuracy: 0.5270 - val_gender_loss: 0.6918 - val_loss: 15.0977
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 130s 208ms/step - age_loss: 13.9048 - age_mae: 13.9048 - gender_accuracy: 0.5218 - gender_loss: 0.6927 - loss: 14.5974 - val_age_loss: 13.2935 - val_age_mae: 13.2970 - val_gender_accuracy: 0.5270 - val_gender_loss: 0.6917 - val_loss: 13.9887
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 126s 202ms/step - age_loss: 12.9335 - age_mae: 12.9335 - gender_accuracy: 0.5218 - gender_loss: 0.6923 - loss: 13.6258 - val_age_loss: 13.3031 - val_age_mae: 13.3059 - val_gender_accuracy: 0.5270 - val_gender_loss: 0.6917 - val_loss: 13.9976
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 125s 201ms/step - age_loss: 12.3854 - age_mae: 12.3854 - gender_accuracy: 0.5218 - gender_loss: 0.6923 - loss: 13.